# Data Cleansing Practice — Part 1
### Zameen.com Real Estate Data

This notebook mirrors a real data-cleaning workflow on scraped property-listing data (`zameenData.csv`) from Zameen.com, in preparation for a house price prediction model.

**How to use this notebook:** each **TODO** cell describes a task. Write your solution in the empty code cell right below it. Run cells in order — later tasks depend on earlier ones. Some tasks include hints in parentheses.


## Setup

**TODO 1:** Import the libraries you'll need: `numpy` as `np`, `pandas` as `pd`, and the built-in `re` module (for regular expressions).

In [ ]:
import numpy as np
import pandas as pd
import re

**TODO 2:** Set pandas display options so that **all** rows and **all** columns are shown when you print a dataframe (`display.max_rows`, `display.max_columns`).

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

**TODO 3:** Load `zameenData.csv` into a dataframe called `df`, using `index_col=0`. The file has a typo in one column name — `pupose` instead of `purpose` — rename it. Display the first 5 rows.

In [ ]:
df = pd.read_csv("zameenData.csv", index_col=0)
df = df.rename(columns={"pupose": "purpose"})     # fix the typo in the column name
df.head()

**TODO 4:** Check the shape of `df`.

In [ ]:
df.shape

## Duplicates

**TODO 5:** Drop duplicate rows from `df`.

In [ ]:
df = df.drop_duplicates()

**TODO 6:** Check the shape again to confirm duplicates were removed.

In [ ]:
df.shape

## Dropping unneeded columns & fixing the `purpose` column

**TODO 7:** Look at the value counts of the `land_type` column, just to get a feel for the categories present.

In [ ]:
df["land_type"].value_counts()

**TODO 8:** Drop the `agent` and `name` columns — they won't be useful for a price-prediction model.

In [ ]:
df = df.drop(columns=["agent", "name"])

**TODO 9:** Look at the value counts of the `purpose` column. Do you see any values besides `"For Rent"` and `"For Sale"`?

In [ ]:
df["purpose"].value_counts()
# Yes, besides "For Rent" and "For Sale" there are wrong values (scraped addresses).

**TODO 10:** Some rows have invalid `purpose` values (a scraping error produced things like addresses instead of "For Rent"/"For Sale"). Select all rows where `purpose` is **neither** `"For Rent"` nor `"For Sale"`, and store them in a variable (e.g. `wrong_scrap`).

In [ ]:
wrong_scrap = df[(df["purpose"] != "For Rent") & (df["purpose"] != "For Sale")]
wrong_scrap

**TODO 11:** Drop the rows found above from `df` (use their index).

In [ ]:
df = df.drop(wrong_scrap.index)

**TODO 12:** Check the `purpose` value counts again — you should now only see `"For Rent"` and `"For Sale"`.

In [ ]:
df["purpose"].value_counts()

**TODO 13:** Keep only rows where `purpose == "For Sale"` (our model is only for sale listings, not rentals). Check the resulting shape.

In [ ]:
df = df[df["purpose"] == "For Sale"].copy()
df.shape

## Price column

**TODO 14:** Look at the top 5 value counts of the raw `price` column. Notice the values look like `"PKR1.2 Crore"` and use **three different units**: Lakh, Crore, and Arab. There is also one bad/garbage value, at index `18903`.

In [ ]:
df["price"].value_counts().head()

**TODO 15:** Write a function `convert_into_crore(text)` that:
- returns `None` if `text` is `None` or not a string,
- extracts the numeric amount and the unit (Lakh / Crore / Arab) from strings like `"PKR1.2 Crore"` using `re.search`,
- converts the amount into **Crore** units (1 Crore = 100 Lakh, 1 Arab = 100 Crore),
- returns the converted numeric value (or `None` if it can't parse the string).

(Hint: `re.search(r"PKR\d+\.?\d*\s*([a-zA-Z]+)", text)` gets you the unit; a similar pattern with `(\d+\.?\d*)` gets you the number.)

In [ ]:
def convert_into_crore(text):
    """Convert a price string such as 'PKR1.2 Crore' into a number in Crore."""
    if text is None or not isinstance(text, str):
        return None

    text = text.replace(",", "")                       # handle values like 'PKR1,200 Lakh'
    unit_match = re.search(r"PKR\s*\d+\.?\d*\s*([a-zA-Z]+)", text)
    number_match = re.search(r"PKR\s*(\d+\.?\d*)", text)

    if unit_match is None or number_match is None:
        return None

    unit = unit_match.group(1).lower()
    amount = float(number_match.group(1))

    if unit == "lakh":
        return amount / 100          # 100 Lakh = 1 Crore
    elif unit == "crore":
        return amount
    elif unit == "arab":
        return amount * 100          # 1 Arab = 100 Crore
    return None

**TODO 16:** Apply `convert_into_crore` to the `price` column and insert the result as a new column called `price_cr` at column position 2 (use `df.insert`).

In [ ]:
df.insert(2, "price_cr", df["price"].apply(convert_into_crore))

**TODO 17:** Preview a random sample of 5 rows to sanity-check the new column.

In [ ]:
df.sample(5)

**TODO 18:** Filter and preview rows where `price` contains `"Arab"`, to double-check your Arab-to-Crore conversion looks right.

In [ ]:
df[df["price"].str.contains("Arab", na=False)]

**TODO 19:** The row at index `18903` has a bad `price_cr` value. Find comparable listings — same `location == "LDA Road, Lahore, Punjab"`, same `land_type == "Residential Plot"`, same `area == "5 Marla"` — and compute the **mean** `price_cr` for that group, rounded to 2 decimals.

In [ ]:
# Comparable listings (the bad row itself is left out so it cannot change the mean)
comparable = df[
    (df["location"] == "LDA Road, Lahore, Punjab")
    & (df["land_type"] == "Residential Plot")
    & (df["area"] == "5 Marla")
].drop(index=18903, errors="ignore")

mean_price = round(comparable["price_cr"].mean(), 2)
mean_price

**TODO 20:** Use `.at[]` to overwrite the bad value at index `18903` in `price_cr` with the mean you just computed.

In [ ]:
df.at[18903, "price_cr"] = mean_price
df.loc[18903]

## Location column

**TODO 21:** Write a function `get_loc(loc)` that strips the trailing `", Lahore, Punjab"` from a location string, returning just the neighborhood/area name.

In [ ]:
def get_loc(loc):
    """Remove the trailing ', Lahore, Punjab' and keep only the area name."""
    if not isinstance(loc, str):
        return None
    return re.sub(r",\s*Lahore,\s*Punjab$", "", loc).strip()

**TODO 22:** Apply `get_loc` to the `location` column and insert the result as a new column called `locat`, at position 4.

In [ ]:
df.insert(4, "locat", df["location"].apply(get_loc))

**TODO 23:** Preview a random sample row to check the result.

In [ ]:
df.sample(1)

**TODO 24:** Check the dataframe shape and the `purpose` value counts again, just to confirm nothing broke.

In [ ]:
print(df.shape)
df["purpose"].value_counts()

## Area column
`Marla`, `Kanal`, `Sq. Yd.`

**TODO 25:** Look at the top 5 value counts of the `area` column.

In [ ]:
df["area"].value_counts().head()

**TODO 26:** Write a function `convert_into_marla(text)` that converts an area string into a single unit, **Marla**:
- `1 Kanal = 20 Marla`
- `1 Sq. Yd. = (value * 9) / 225` Marla
- values already in Marla stay the same

(Hint: use regex to separate the numeric value from the unit text, e.g. `re.search(r"\d+\.?\d*", text)` for the number.)

In [ ]:
def convert_into_marla(text):
    """Convert an area string into Marla."""
    if not isinstance(text, str):
        return None

    number_match = re.search(r"\d+\.?\d*", text.replace(",", ""))
    if number_match is None:
        return None
    value = float(number_match.group())

    if "Kanal" in text:
        return value * 20                  # 1 Kanal = 20 Marla
    elif "Sq. Yd." in text:
        return (value * 9) / 225           # Sq. Yd. to Marla
    elif "Marla" in text:
        return value                       # already in Marla
    return None

**TODO 27:** Apply `convert_into_marla` to `area` and insert the result as a new column `area_value`, at position 7.

In [ ]:
df.insert(7, "area_value", df["area"].apply(convert_into_marla))

**TODO 28:** Filter and preview rows where `area` contains `"Sq. Yd."`, to check your conversion looks correct.

In [ ]:
df[df["area"].str.contains("Sq. Yd.", na=False, regex=False)].head(10)

**TODO 29:** This dataset should only contain Lahore properties. Find rows where `location` contains `"Karachi"` and drop them from `df`.

In [ ]:
karachi_rows = df[df["location"].str.contains("Karachi", na=False)]
df = df.drop(karachi_rows.index)
df.shape

## Baths column

**TODO 30:** Check the value counts of `baths`.

In [ ]:
df["baths"].value_counts()

**TODO 31:** Replace `"-"` values in `baths` with `"0"`.

In [ ]:
df["baths"] = df["baths"].replace("-", "0")

**TODO 32:** Preview a random sample row.

In [ ]:
df.sample(1)

## Bed column

**TODO 33:** Check the value counts of `bed`.

In [ ]:
df["bed"].value_counts()

**TODO 34:** Replace `"-"` values in `bed` with `"0"`.

In [ ]:
df["bed"] = df["bed"].replace("-", "0")

**TODO 35:** Preview the columns `land_type`, `area_value`, `price_cr`, and `baths` together, just to sanity-check everything so far.

In [ ]:
df[["land_type", "area_value", "price_cr", "baths"]].head(10)

## Land type column

**TODO 36:** Check the value counts of `land_type` again.

In [ ]:
df["land_type"].value_counts()

**TODO 37:** Convert the `area_value` column to `float` dtype.

In [ ]:
df["area_value"] = df["area_value"].astype(float)

**TODO 38:** Check `land_type` value counts once more.

In [ ]:
df["land_type"].value_counts()

**TODO 39:** For rows where `land_type == "Building"`, compute the overall price-per-marla: sum of `price_cr` divided by sum of `area_value`.

In [ ]:
building = df[df["land_type"] == "Building"]
price_per_marla = building["price_cr"].sum() / building["area_value"].sum()
price_per_marla

**TODO 40:** Define a list `useful_land_types` containing: `'House'`, `'Flat'`, `'Building'`, `'Residential Plot'`, `'Commercial Plot'`, `'Plot File'`, `'Agricultural Land'`, `'Industrial Land'`, `'Factory'`, `'Farm House'`, `'Shop'`, `'Office'`. Keep only the rows whose `land_type` is in this list (this drops rare/tiny categories). Check the value counts of `land_type` afterward.

In [ ]:
useful_land_types = [
    "House", "Flat", "Building", "Residential Plot", "Commercial Plot", "Plot File",
    "Agricultural Land", "Industrial Land", "Factory", "Farm House", "Shop", "Office",
]
df = df[df["land_type"].isin(useful_land_types)].copy()
df["land_type"].value_counts()

## Save the cleaned data

**TODO 41:** Save the cleaned dataframe to `dataCleansing1_complete.csv`.

In [ ]:
df.to_csv("dataCleansing1_complete.csv")     # index is kept, because Part 2 loads it with index_col=0